### FASE EXTRACT: Setup & Data Loading
**Tujuan:**
Melakukan autentikasi API Kaggle, mengunduh dataset raw, dan memuatnya ke dalam DataFrame Pandas. Sistem logging diaktifkan untuk melacak rekam jejak berjalannya ETL.

In [ ]:
# 1. Install library pendukung
!pip install -q kaggle pandas

from google.colab import files
import os
import pandas as pd
import numpy as np
import re
import glob
import logging
from IPython.display import display

# Untuk mencetak Logging tambah force=True
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - [ETL LOG] %(message)s', force=True)

print("=== AUTENTIKASI KAGGLE ===")
print("Silakan unggah file kaggle.json Anda: ")
uploaded = files.upload()

# Memindahkan file konfigurasi ke folder tersembunyi Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

logging.info("Setup Kaggle API berhasil.")

print("=== FASE EXTRACT: Mengunduh Dataset dari Kaggle ===")
!kaggle datasets download -d sdeogade/sparse-industrial-machine-time-series-dataset
!unzip -q sparse-industrial-machine-time-series-dataset.zip -d dataset_tambang

csv_file = glob.glob('dataset_tambang/*.csv')[0]
df_raw = pd.read_csv(csv_file, nrows=50000)

logging.info(f"Fase Extract Selesai. Total baris data yang dimuat: {len(df_raw)}")
print("\n--- TAMPILAN DATA MENTAH (BEFORE) ---")
display(df_raw.head(5))

=== AUTENTIKASI KAGGLE ===
Silakan unggah file kaggle.json Anda: 


Saving kaggle.json to kaggle.json


2026-07-01 05:05:49,267 - INFO - [ETL LOG] Setup Kaggle API berhasil.


=== FASE EXTRACT: Mengunduh Dataset dari Kaggle ===
Dataset URL: https://www.kaggle.com/datasets/sdeogade/sparse-industrial-machine-time-series-dataset
License(s): CC-BY-SA-4.0
100% 4.49M/4.49M [00:00<00:00, 74.0MB/s]



2026-07-01 05:05:51,521 - INFO - [ETL LOG] Fase Extract Selesai. Total baris data yang dimuat: 50000



--- TAMPILAN DATA MENTAH (BEFORE) ---


,transaction_date,asset_tag,machine_type,plant_code,part_no,part_description,part_family,criticality,uom,unit_cost_inr,...,temp_bearing_degC,temp_motor_degC,vibration_h_mms,vibration_v_mms,oil_pressure_bar,load_pct,shaft_rpm,power_consumption_kw,breakdown_flag,wo_type
0,2022-01-03,AST-1041,CNC Lathe,PUN-01,MRO-10045,Deep Groove Ball Bearing 6205-2RS,Bearing,A,EA,850,...,63.3,63.5,4.944,3.886,3.71,85.9,1794,16.09,0,NaN
1,2022-01-04,AST-1041,CNC Lathe,PUN-01,MRO-10045,Deep Groove Ball Bearing 6205-2RS,Bearing,A,EA,850,...,64.0,64.8,4.614,3.530,3.67,82.0,1790,15.53,0,NaN
2,2022-01-05,AST-1041,CNC Lathe,PUN-01,MRO-10045,Deep Groove Ball Bearing 6205-2RS,Bearing,A,EA,850,...,64.4,66.7,4.360,2.976,3.74,83.0,1827,16.20,0,NaN
3,2022-01-06,AST-1041,CNC Lathe,PUN-01,MRO-10045,Deep Groove Ball Bearing 6205-2RS,Bearing,A,EA,850,...,64.5,68.1,4.522,3.315,3.90,82.9,1811,16.47,0,NaN
4,2022-01-07,AST-1041,CNC Lathe,PUN-01,MRO-10045,Deep Groove Ball Bearing 6205-2RS,Bearing,A,EA,850,...,62.2,66.3,4.340,3.434,3.71,80.1,1810,15.08,0,NaN


### FASE TRANSFORM 1: Formatting & Regex Extraction
**Tujuan:**
1. Mengubah format `transaction_date` (String) menjadi Integer ID (`YYYYMMDD`) agar hierarki dimensi waktu dapat dibaca oleh Power BI.
2. Menggabungkan getaran horizontal dan vertikal menjadi satu metrik tunggal.
3. Menggunakan **Regex** untuk mengekstrak angka murni dari `asset_tag` untuk efisiensi memori.

In [ ]:
logging.info("=== FASE TRANSFORM 1: Formatting & Regex ===")
df_clean = df_raw.copy()

# 1. Konversi format waktu menjadi Integer ID
df_clean['transaction_date'] = pd.to_datetime(df_clean['transaction_date'])
df_clean['Date_ID'] = df_clean['transaction_date'].dt.strftime('%Y%m%d').astype(int)

# 2. Integration: Menjumlahkan metrik getaran
df_clean['total_vibration_mms'] = df_clean['vibration_h_mms'] + df_clean['vibration_v_mms']

# 3. Text Parsing (Regex): Mengekstrak angka dari asset_tag
def extract_asset_number(tag):
    match = re.search(r'-(\d+)', str(tag))
    return match.group(1) if match else "Unknown"

df_clean['asset_number'] = df_clean['asset_tag'].apply(extract_asset_number)
logging.info("Formatting dan ekstraksi Regex berhasil dieksekusi.")

print("\n--- HASIL FORMATTING & REGEX ---")
display(df_clean[['transaction_date', 'Date_ID', 'asset_tag', 'asset_number', 'total_vibration_mms']].head(5))

2026-07-01 05:06:06,968 - INFO - [ETL LOG] === FASE TRANSFORM 1: Formatting & Regex ===
2026-07-01 05:06:07,241 - INFO - [ETL LOG] Formatting dan ekstraksi Regex berhasil dieksekusi.



--- HASIL FORMATTING & REGEX ---


,transaction_date,Date_ID,asset_tag,asset_number,total_vibration_mms
0,2022-01-03,20220103,AST-1041,1041,8.830
1,2022-01-04,20220104,AST-1041,1041,8.144
2,2022-01-05,20220105,AST-1041,1041,7.336
3,2022-01-06,20220106,AST-1041,1041,7.837
4,2022-01-07,20220107,AST-1041,1041,7.774


### FASE TRANSFORM 2: Feature Engineering
**Tujuan:**
Sesuai dengan target analitik proses bisnis telemetri, metrik batas aman operasional mesin tidak tersedia di data raw. Menggunakan fungsi Python, mesin diklasifikasikan menjadi "Critical Stress", "High Load", atau "Normal Operation" berdasarkan korelasi suhu dan beban kerja.

In [ ]:
logging.info("=== FASE TRANSFORM 2: Feature Engineering ===")

# Logika bersyarat untuk mendeteksi anomali operasional mesin
def get_stress_level(row):
    if row['load_pct'] > 85 and row['temp_bearing_degC'] > 80:
        return 'Critical Stress'
    elif row['load_pct'] > 75:
        return 'High Load'
    else:
        return 'Normal Operation'

df_clean['Machine_Stress_Level'] = df_clean.apply(get_stress_level, axis=1)
logging.info("Kolom analitik 'Machine_Stress_Level' siap digunakan.")

print("\n--- DISTRIBUSI KATEGORI STRES MESIN ---")
display(df_clean[['asset_tag', 'load_pct', 'temp_bearing_degC', 'Machine_Stress_Level']].head(5))

2026-07-01 05:06:33,522 - INFO - [ETL LOG] === FASE TRANSFORM 2: Feature Engineering ===
2026-07-01 05:06:33,902 - INFO - [ETL LOG] Kolom analitik 'Machine_Stress_Level' siap digunakan.



--- DISTRIBUSI KATEGORI STRES MESIN ---


,asset_tag,load_pct,temp_bearing_degC,Machine_Stress_Level
0,AST-1041,85.9,63.3,High Load
1,AST-1041,82.0,64.0,High Load
2,AST-1041,83.0,64.4,High Load
3,AST-1041,82.9,64.5,High Load
4,AST-1041,80.1,62.2,High Load


### FASE TRANSFORM 3: Pemecahan 3 Star Schema
**Tujuan:**
Memecah flat file menjadi 4 Tabel Dimensi (Date, Plant, Machine, Part) dan 3 Tabel Fakta terpisah (Demand, Telemetry, Breakdown) untuk membentuk **3 Star Schema** yang independen pada arsitektur Data Warehouse.

In [ ]:
logging.info("=== FASE TRANSFORM 3: Pembuatan 3 Star Schema Independen ===")

# --- 1. MEMBANGUN 4 TABEL DIMENSI (Master Data) ---
# Mengurutkan data berdasarkan tanggal terbaru (descending) untuk mendapatkan status mutakhir mesin (SCD Type 1)
dim_machine = df_clean.sort_values('transaction_date', ascending=False)[['asset_tag', 'asset_number', 'machine_type', 'criticality']].drop_duplicates(subset=['asset_tag'], keep='first').reset_index(drop=True)
dim_part = df_clean[['part_no', 'part_family', 'unit_cost_inr', 'uom']].drop_duplicates(subset=['part_no']).reset_index(drop=True)

dim_plant = df_clean[['plant_code']].drop_duplicates(subset=['plant_code']).reset_index(drop=True)
dim_plant['plant_name'] = "Plant " + dim_plant['plant_code'].astype(str)

dim_date = df_clean[['Date_ID', 'transaction_date']].drop_duplicates(subset=['Date_ID']).sort_values('Date_ID').reset_index(drop=True)
dim_date['Year'] = dim_date['transaction_date'].dt.year
dim_date['Month'] = dim_date['transaction_date'].dt.month
dim_date['Day'] = dim_date['transaction_date'].dt.day

# --- 2. MEMBANGUN 3 TABEL FAKTA (Mewakili 3 Star Schema) ---

# Fakta untuk Star Schema 1: Logistik (Filter qty_issued > 0 untuk sparsity cleaning)
fact_demand = df_clean[df_clean['qty_issued'] > 0][
    ['Date_ID', 'asset_tag', 'part_no', 'plant_code', 'qty_issued', 'issue_value_inr']
].reset_index(drop=True)

# Fakta untuk Star Schema 2: Telemetri
fact_telemetry = df_clean[
    ['Date_ID', 'asset_tag', 'plant_code', 'temp_bearing_degC', 'total_vibration_mms', 'load_pct', 'Machine_Stress_Level']
].reset_index(drop=True)

# Fakta untuk Star Schema 3: Insiden / Breakdown
fact_breakdown = df_clean[
    ['Date_ID', 'asset_tag', 'plant_code', 'breakdown_flag', 'wo_type']
].reset_index(drop=True)

logging.info(f"Structuring 3 Star Schema berhasil! Fact_Demand tersisa {len(fact_demand)} transaksi riil.")

# [PERBAIKAN 2]: Menampilkan ke-4 Tabel Dimensi dan ke-3 Tabel Fakta untuk di-screenshot
print("\n========================================================")
print("--- TAMPILAN 4 TABEL DIMENSI (AFTER) ---")
print("1. Dim_Machine:")
display(dim_machine.head(5))
print("\n2. Dim_Part:")
display(dim_part.head(5))
print("\n3. Dim_Plant:")
display(dim_plant.head(5))
print("\n4. Dim_Date:")
display(dim_date.head(5))

print("\n========================================================")
print("--- TAMPILAN 3 TABEL FAKTA (AFTER) ---")
print("1. Fact_Demand:")
display(fact_demand.head(5))
print("\n2. Fact_Telemetry:")
display(fact_telemetry.head(5))
print("\n3. Fact_Breakdown:")
display(fact_breakdown.head(5))
print("========================================================\n")

2026-07-01 05:07:17,787 - INFO - [ETL LOG] === FASE TRANSFORM 3: Pembuatan 3 Star Schema Independen ===
2026-07-01 05:07:17,841 - INFO - [ETL LOG] Structuring 3 Star Schema berhasil! Fact_Demand tersisa 4573 transaksi riil.



--- TAMPILAN 4 TABEL DIMENSI (AFTER) ---
1. Dim_Machine:


,asset_tag,asset_number,machine_type,criticality
0,AST-1041,1041,CNC Lathe,C
1,AST-1042,1042,CNC Lathe,C
2,AST-2017,2017,Hydraulic Press,A



2. Dim_Part:


,part_no,part_family,unit_cost_inr,uom
0,MRO-10045,Bearing,850,EA
1,MRO-10046,Bearing,2100,EA
2,MRO-10047,Bearing,1450,EA
3,MRO-20031,Seal & Gasket,3200,KIT
4,MRO-20032,Seal & Gasket,12,EA


...

,Date_ID,asset_tag,part_no,plant_code,qty_issued,issue_value_inr
0,20220106,AST-1041,MRO-10045,PUN-01,1,850
1,20220118,AST-1041,MRO-10045,PUN-01,1,850
2,20220119,AST-1041,MRO-10045,PUN-01,1,850
3,20220128,AST-1041,MRO-10045,PUN-01,1,850
4,20220207,AST-1041,MRO-10045,PUN-01,1,850



2. Fact_Telemetry:


,Date_ID,asset_tag,plant_code,temp_bearing_degC,total_vibration_mms,load_pct,Machine_Stress_Level
0,20220103,AST-1041,PUN-01,63.3,8.830,85.9,High Load
1,20220104,AST-1041,PUN-01,64.0,8.144,82.0,High Load
2,20220105,AST-1041,PUN-01,64.4,7.336,83.0,High Load
3,20220106,AST-1041,PUN-01,64.5,7.837,82.9,High Load
4,20220107,AST-1041,PUN-01,62.2,7.774,80.1,High Load



3. Fact_Breakdown:


,Date_ID,asset_tag,plant_code,breakdown_flag,wo_type
0,20220103,AST-1041,PUN-01,0,NaN
1,20220104,AST-1041,PUN-01,0,NaN
2,20220105,AST-1041,PUN-01,0,NaN
3,20220106,AST-1041,PUN-01,0,NaN
4,20220107,AST-1041,PUN-01,0,NaN


### FASE LOAD: Ekspor ke Power BI
**Tujuan:**
Mengekspor 7 komponen tabel (4 Dimensi + 3 Fakta) ke dalam format `.csv` datar agar siap diimpor ke dalam Power BI untuk visualisasi visual 3 Star Schema.

In [ ]:
logging.info("=== FASE LOAD: Ekspor File CSV untuk Power BI ===")

# Menyimpan 4 Tabel Dimensi
dim_machine.to_csv("Dim_Machine.csv", index=False)
dim_part.to_csv("Dim_Part.csv", index=False)
dim_plant.to_csv("Dim_Plant.csv", index=False)
dim_date.to_csv("Dim_Date.csv", index=False)

# Menyimpan 3 Tabel Fakta
fact_demand.to_csv("Fact_Demand.csv", index=False)
fact_telemetry.to_csv("Fact_Telemetry.csv", index=False)
fact_breakdown.to_csv("Fact_Breakdown.csv", index=False)

logging.info("PROSES ETL SELESAI. 7 File CSV siap diimpor ke Power BI untuk membentuk 3 Star Schema.")

2026-07-01 05:07:43,180 - INFO - [ETL LOG] === FASE LOAD: Ekspor File CSV untuk Power BI ===
2026-07-01 05:07:44,258 - INFO - [ETL LOG] PROSES ETL SELESAI. 7 File CSV siap diimpor ke Power BI untuk membentuk 3 Star Schema.
